In [2]:
# ╔══════════════════════════════════════════════════════╗
# ║  Cell 1 — Bootstrap                                  ║
# ╚══════════════════════════════════════════════════════╝
import sys
sys.path.append("..")          # make src/ visible from notebooks/

from src.phase1_data_pipeline import run_pipeline, load_configs
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# ╔══════════════════════════════════════════════════════╗
# ║  Cell 2 — Run pipeline                               ║
# ╚══════════════════════════════════════════════════════╝
splits, master, reports = run_pipeline(
    data_cfg_path     = "../config/data_config.json",
    pipeline_cfg_path = "../config/pipeline_config.json",
    verbose           = True
)


────────────────────────────────────────────────────────────
  PHASE 1 — DATA PIPELINE
────────────────────────────────────────────────────────────

[·] Loading & cleaning target


FileNotFoundError: Data file not found: data/EURUSD_M1.csv

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  Cell 3 — Master DataFrame overview                  ║
# ╚══════════════════════════════════════════════════════╝
print(f"Shape   : {master.shape}")
print(f"Columns : {list(master.columns)}\n")
master.head(3)

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  Cell 4 — Split sizes                                ║
# ╚══════════════════════════════════════════════════════╝
for name, df in splits.items():
    print(f"{name:<12}: {len(df):>8,} bars   "
          f"{df.index[0]}  →  {df.index[-1]}")

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  Cell 5 — Cleaning report                            ║
# ╚══════════════════════════════════════════════════════╝
pd.DataFrame(reports).T

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  Cell 6 — NaN audit (session breaks)                 ║
# ╚══════════════════════════════════════════════════════╝
nan_counts = master.isna().sum()
nan_counts = nan_counts[nan_counts > 0]

if nan_counts.empty:
    print("No NaN values — all gaps were within max_fill_gap.")
else:
    print("Columns with NaN (gaps > max_fill_gap = session breaks):")
    print(nan_counts.to_string())

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  Cell 7 — Forward-fill audit                         ║
# ╚══════════════════════════════════════════════════════╝
fill_cols = [c for c in master.columns if c.endswith("_is_filled")]
fill_totals = master[fill_cols].sum().rename("filled_bars")
fill_pct    = (fill_totals / len(master) * 100).rename("fill_%")
pd.concat([fill_totals, fill_pct], axis=1).round(2)

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  Cell 8 — Session distribution per split             ║
# ╚══════════════════════════════════════════════════════╝
_, pipeline_cfg = load_configs(
    "../config/data_config.json",
    "../config/pipeline_config.json"
)
session_names = {
    int(k): v["name"]
    for k, v in pipeline_cfg["sessions"].items()
}

for name, df in splits.items():
    dist = (
        df["session"]
        .value_counts()
        .sort_index()
        .rename(index=session_names)
    )
    print(f"\n── {name} ──────────────────")
    print(dist.to_string())

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  Cell 9 — Sanity plot                                ║
# ╚══════════════════════════════════════════════════════╝
COLORS = {"train": "green", "validation": "orange", "test": "red"}

fig, ax = plt.subplots(figsize=(15, 3))
master["close"].plot(ax=ax, lw=0.4, color="steelblue", label="Target close")

for name, df in splits.items():
    ax.axvspan(
        df.index[0], df.index[-1],
        alpha=0.12,
        color=COLORS[name],
        label=name
    )

ax.set_title("Target Close — Train / Validation / Test regions")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()